# Práctica 2 · Similitud de documentos con Bag-of-Words y similitud del coseno

El objetivo de esta práctica es representar un conjunto de documentos mediante el modelo Bag-of-Words (BoW) y calcular la similitud del coseno entre sus representaciones vectoriales.

Los 12 documentos utilizados tratan principalmente dos temas: tecnología móvil y vehículos eléctricos. Algunos documentos se centran exclusivamente en uno de estos temas, mientras que otros combinan ambos. A partir de las similitudes obtenidas se analizará si los documentos que tratan temas relacionados presentan también un mayor parecido en su vocabulario.

## 1. Importación de librerías

Se utilizan las siguientes librerías:

- `python-docx` para leer el contenido de los documentos.
- `nltk` para eliminar las palabras vacías del español.
- `re` para realizar el preprocesamiento del texto.
- `torch` para construir los vectores Bag-of-Words y calcular la similitud del coseno.

In [1]:
from pathlib import Path
from docx import Document
from nltk.corpus import stopwords
import re
import nltk
import torch
import torch.nn.functional as F


## 2. Carga de documentos

Los documentos se encuentran almacenados en archivos `.docx` denominados `doc01.docx` hasta `doc12.docx`.

En primer lugar, se comprueba que los 12 archivos están disponibles y que contienen texto. Posteriormente, se extrae el contenido de cada documento para utilizarlo en las siguientes etapas del análisis.

In [2]:
carpeta = Path.cwd()
archivos = sorted(carpeta.glob("doc*.docx"))

esperados = [f"doc{i:02d}.docx" for i in range(1, 13)]
nombres_encontrados = [ruta.name for ruta in archivos]
if nombres_encontrados != esperados:
    raise FileNotFoundError(
        f"Se esperaban los archivos {esperados} en {carpeta}. "
        f"Se encontraron: {nombres_encontrados}."
    )

documentos = []
nombres = []
for ruta in archivos:
    documento_word = Document(ruta)
    texto = " ".join(
        parrafo.text for parrafo in documento_word.paragraphs
        if parrafo.text.strip()
    )
    if not texto.strip():
        raise ValueError(f"El documento {ruta.name} no contiene texto en párrafos.")
    documentos.append(texto)
    nombres.append(ruta.name)

print(f"Documentos cargados ({len(nombres)}): {nombres}")

Documentos cargados (12): ['doc01.docx', 'doc02.docx', 'doc03.docx', 'doc04.docx', 'doc05.docx', 'doc06.docx', 'doc07.docx', 'doc08.docx', 'doc09.docx', 'doc10.docx', 'doc11.docx', 'doc12.docx']


## 3. Preprocesamiento

El preprocesamiento se realiza en varias etapas. Primero, cada documento se convierte a minúsculas. Después se eliminan los signos de puntuación y los caracteres que no pertenecen al alfabeto. A continuación, el texto se divide en palabras mediante una tokenización basada en espacios.

También se eliminan las palabras vacías (*stop words*) del español. Esta última etapa es opcional en el enunciado y se utiliza para reducir la influencia de palabras muy frecuentes que aportan poca información para distinguir los documentos.

In [3]:
nltk.download("stopwords", quiet=True)

STOPWORDS = set(stopwords.words("spanish"))

def preprocesar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto)
    palabras = texto.split()
    palabras = [p for p in palabras if p not in STOPWORDS]
    return palabras

docs_tokenizados = [preprocesar(doc) for doc in documentos]

for nombre, tokens in zip(nombres, docs_tokenizados):
    print(f"{nombre}: {len(tokens)} palabras tras preprocesado")

doc01.docx: 58 palabras tras preprocesado
doc02.docx: 51 palabras tras preprocesado
doc03.docx: 47 palabras tras preprocesado
doc04.docx: 44 palabras tras preprocesado
doc05.docx: 42 palabras tras preprocesado
doc06.docx: 46 palabras tras preprocesado
doc07.docx: 44 palabras tras preprocesado
doc08.docx: 43 palabras tras preprocesado
doc09.docx: 45 palabras tras preprocesado
doc10.docx: 42 palabras tras preprocesado
doc11.docx: 74 palabras tras preprocesado
doc12.docx: 83 palabras tras preprocesado


## 4. Construcción de Bag-of-Words

A partir de los tokens obtenidos en el preprocesamiento se construye un vocabulario global con todas las palabras diferentes presentes en los 12 documentos.

Cada documento se representa mediante un vector cuya longitud coincide con el número de términos del vocabulario. En cada posición se almacena la frecuencia con la que aparece el término correspondiente en ese documento.

Finalmente, los vectores se agrupan en una matriz en la que cada fila representa un documento y cada columna representa un término del vocabulario.

In [4]:
vocabulario = sorted(set(palabra for tokens in docs_tokenizados for palabra in tokens))
palabra_a_indice = {palabra: i for i, palabra in enumerate(vocabulario)}

print(f"Tamaño del vocabulario: {len(vocabulario)} palabras")

def vectorizar_bow(tokens, vocab_index):
    vector = torch.zeros(len(vocab_index))
    for palabra in tokens:
        if palabra in vocab_index:
            vector[vocab_index[palabra]] += 1
    return vector

matriz_bow = torch.stack([vectorizar_bow(tokens, palabra_a_indice) for tokens in docs_tokenizados])
print(f"Forma de la matriz BoW: {matriz_bow.shape}")

Tamaño del vocabulario: 411 palabras
Forma de la matriz BoW: torch.Size([12, 411])


## 5. Matriz de similitud del coseno

La similitud entre dos documentos se calcula mediante la similitud del coseno aplicada a sus vectores Bag-of-Words:

$$
\operatorname{sim}(A,B)=
\frac{A\cdot B}
{\lVert A\rVert\,\lVert B\rVert}
$$

Un valor cercano a 1 indica una alta similitud en la representación vectorial, mientras que un valor cercano a 0 indica que los documentos comparten poco o ningún vocabulario después del preprocesamiento. En este caso, al utilizar vectores BoW con frecuencias no negativas, los valores obtenidos se encuentran entre 0 y 1.

La similitud se calcula para todos los pares de documentos. La diagonal de la matriz toma el valor 1, ya que cada documento es idéntico a sí mismo, y la matriz resultante es simétrica.

In [5]:
n_docs = matriz_bow.shape[0]
similitud_bow = torch.zeros((n_docs, n_docs))

for i in range(n_docs):
    for j in range(n_docs):
        similitud_bow[i, j] = F.cosine_similarity(
            matriz_bow[i].unsqueeze(0), matriz_bow[j].unsqueeze(0)
        )

print("Matriz de similitud del coseno (BoW):")
for i in range(n_docs):
    fila = " ".join(f"{similitud_bow[i, j]:.2f}" for j in range(n_docs))
    print(f"{nombres[i]}: {fila}")

Matriz de similitud del coseno (BoW):
doc01.docx: 1.00 0.04 0.00 0.08 0.10 0.14 0.06 0.02 0.00 0.05 0.15 0.09
doc02.docx: 0.04 1.00 0.10 0.00 0.00 0.10 0.12 0.03 0.00 0.11 0.19 0.11
doc03.docx: 0.00 0.10 1.00 0.00 0.00 0.03 0.10 0.02 0.00 0.09 0.00 0.05
doc04.docx: 0.08 0.00 0.00 1.00 0.10 0.07 0.06 0.13 0.22 0.00 0.14 0.25
doc05.docx: 0.10 0.00 0.00 0.10 1.00 0.06 0.00 0.19 0.21 0.04 0.11 0.06
doc06.docx: 0.14 0.10 0.03 0.07 0.06 1.00 0.13 0.04 0.02 0.08 0.13 0.10
doc07.docx: 0.06 0.12 0.10 0.06 0.00 0.13 1.00 0.02 0.00 0.06 0.19 0.15
doc08.docx: 0.02 0.03 0.02 0.13 0.19 0.04 0.02 1.00 0.09 0.02 0.09 0.03
doc09.docx: 0.00 0.00 0.00 0.22 0.21 0.02 0.00 0.09 1.00 0.04 0.17 0.32
doc10.docx: 0.05 0.11 0.09 0.00 0.04 0.08 0.06 0.02 0.04 1.00 0.07 0.09
doc11.docx: 0.15 0.19 0.00 0.14 0.11 0.13 0.19 0.09 0.17 0.07 1.00 0.30
doc12.docx: 0.09 0.11 0.05 0.25 0.06 0.10 0.15 0.03 0.32 0.09 0.30 1.00


## 6. Análisis de las similitudes

A continuación se comparan los documentos entre sí para identificar las parejas con mayor y menor similitud.

Para evitar repetir cada comparación dos veces, únicamente se consideran las parejas en las que `i < j`. Tampoco se incluyen las comparaciones de cada documento consigo mismo.

In [6]:
pares = []
for i in range(n_docs):
    for j in range(i + 1, n_docs):
        pares.append((similitud_bow[i, j].item(), nombres[i], nombres[j]))

pares_ordenados = sorted(pares, key=lambda x: x[0], reverse=True)

print("Pares más similares (BoW):")
for sim, a, b in pares_ordenados[:5]:
    print(f"  {a} - {b}: {sim:.3f}")

print("Pares menos similares (BoW):")
for sim, a, b in pares_ordenados[-5:]:
    print(f"  {a} - {b}: {sim:.3f}")

Pares más similares (BoW):
  doc09.docx - doc12.docx: 0.323
  doc11.docx - doc12.docx: 0.302
  doc04.docx - doc12.docx: 0.249
  doc04.docx - doc09.docx: 0.217
  doc05.docx - doc09.docx: 0.207
Pares menos similares (BoW):
  doc03.docx - doc09.docx: 0.000
  doc03.docx - doc11.docx: 0.000
  doc04.docx - doc10.docx: 0.000
  doc05.docx - doc07.docx: 0.000
  doc07.docx - doc09.docx: 0.000


### Parejas menos similares

Las cinco parejas mostradas como menos similares presentan una similitud de **0,000**. Esto significa que no comparten ningún término después del preprocesamiento y, por tanto, su producto escalar es 0.

Hay otras parejas que también presentan este valor, por lo que los cinco pares mostrados constituyen una selección entre varias parejas empatadas.

### Relación entre similitud y contenido

Para interpretar los resultados se comparan las parejas con mayor similitud con el contenido de los documentos y con los términos que tienen en común.

Los 12 documentos se pueden agrupar, de forma general, en tres grupos:

- **Tecnología móvil:** `doc01`, `doc02`, `doc03`, `doc06`, `doc07` y `doc10`.
- **Vehículos eléctricos:** `doc04`, `doc05`, `doc08` y `doc09`.
- **Ambos ámbitos:** `doc11` y `doc12`.

Esta clasificación permite comprobar si la similitud léxica obtenida mediante Bag-of-Words guarda relación con los temas tratados.

In [7]:
for nombre, texto in zip(nombres, documentos):
    extracto = " ".join(texto.split())[:300]
    print(f"{nombre}: {extracto}{'…' if len(texto) > 300 else ''}\n")

print("Términos compartidos en los cinco pares más similares:")
for sim, a, b in pares_ordenados[:5]:
    ia, ib = nombres.index(a), nombres.index(b)
    comunes = set(docs_tokenizados[ia]) & set(docs_tokenizados[ib])
    ordenados = sorted(
        comunes,
        key=lambda p: (-(docs_tokenizados[ia].count(p) * docs_tokenizados[ib].count(p)), p)
    )
    print(f"{a} – {b} ({sim:.3f}): {', '.join(ordenados[:15])}")

doc01.docx: Nuevo smartphone de alta gama promete revolucionar el mercado Hoy se anunció el lanzamiento del último smartphone de la reconocida marca Tecnophone. Este nuevo dispositivo, denominado Tecnophone XZ, promete características sobresalientes que seguramente causarán revuelo en la industria móvil. Incorp…

doc02.docx: La duración de la batería en smartphones: un desafío constante La vida útil de la batería se ha convertido en uno de los temas más discutidos en el mundo de los smartphones. Con la constante demanda de aplicaciones y servicios que consumen grandes cantidades de energía, los fabricantes están bajo pr…

doc03.docx: Las aplicaciones móviles más descargadas del mes El auge de las aplicaciones móviles continúa creciendo a pasos agigantados. Este mes, la aplicación de edición de video ClipMaster ha liderado las listas de descargas, seguida de cerca por la app de fitness FitJourney. Ambas aplicaciones demuestran el…

doc04.docx: El futuro de los coches eléctricos brilla m

### Interpretación y conclusiones

Los resultados muestran que las parejas con mayor similitud presentan, en general, contenidos relacionados y comparten parte de su vocabulario.

El par con mayor similitud es `doc09`–`doc12`, con un valor de **0,323**. El `doc09` trata sobre los supercargadores y la carga rápida de vehículos eléctricos, mientras que el `doc12` presenta una aplicación móvil destinada a localizar, reservar y utilizar puntos de carga para estos vehículos. Ambos documentos comparten términos como `carga`, `eléctricos`, `vehículos`, `conductores`, `cargar` y `coches`. Por tanto, el resultado es coherente con el contenido de los documentos.

El segundo par con mayor similitud es `doc11`–`doc12`, con un valor de **0,302**. En este caso, ambos documentos combinan los dos temas principales del corpus: la tecnología móvil y los vehículos eléctricos. El `doc11` describe la integración de un smartphone con un vehículo eléctrico, mientras que el `doc12` presenta una aplicación móvil relacionada con la carga de vehículos eléctricos. Entre las palabras compartidas aparecen `móvil`, `carga`, `conducción`, `conductores`, `eléctricos`, `vehículos`, `navegación`, `smartphones` y `tecnología`.

También aparecen entre las parejas con mayor similitud `doc04`–`doc12` (0,249), `doc04`–`doc09` (0,217) y `doc05`–`doc09` (0,207). Estos documentos están relacionados con los vehículos eléctricos y comparten vocabulario específico de este ámbito. Por ejemplo, `doc04` trata sobre un nuevo coche eléctrico y sus estaciones de carga, `doc05` sobre la adopción de vehículos eléctricos y `doc09` sobre la carga rápida.

Los documentos relacionados exclusivamente con tecnología móvil también presentan similitudes entre sí, aunque no siempre aparecen entre las cinco parejas con mayor valor. Por ejemplo, `doc01` trata sobre un nuevo smartphone, `doc02` sobre la duración de la batería, `doc03` sobre aplicaciones móviles, `doc06` sobre pantallas plegables, `doc07` sobre realidad aumentada y `doc10` sobre seguridad móvil. Aunque todos pertenecen al mismo ámbito general, sus contenidos son diferentes y, por tanto, no siempre comparten suficiente vocabulario como para obtener valores elevados de similitud.

En el extremo contrario, varias parejas presentan una similitud de **0,000**. Por ejemplo, `doc03`–`doc09`, `doc03`–`doc11`, `doc04`–`doc10`, `doc05`–`doc07` y `doc07`–`doc09`. En estos casos no existe ninguna palabra común después del preprocesamiento. Esto es coherente con el hecho de que algunos de estos documentos pertenecen a ámbitos diferentes, como aplicaciones móviles y carga de vehículos eléctricos. Sin embargo, un valor de 0 no significa que los documentos sean necesariamente opuestos o que no exista ninguna relación entre sus contenidos; únicamente indica que no comparten vocabulario en la representación utilizada.

En conjunto, los resultados muestran que la similitud del coseno aplicada a Bag-of-Words permite detectar cierta relación entre el vocabulario de los documentos y los temas que tratan. Los documentos que comparten conceptos y términos específicos, especialmente en torno a los vehículos eléctricos y su sistema de carga, tienden a presentar valores de similitud más altos.

Por otro lado, el método tiene algunas limitaciones. Bag-of-Words no tiene en cuenta el orden de las palabras ni su significado, y considera diferentes palabras las distintas formas de un mismo término. Además, dos documentos pueden tratar el mismo tema utilizando vocabulario diferente y obtener una similitud baja. Por tanto, la similitud obtenida representa principalmente una medida de parecido léxico y no una medida completa de similitud semántica.

## 7. Representación TF-IDF

Además de la representación Bag-of-Words, se calcula una representación TF-IDF de los documentos para comparar los resultados.

TF-IDF combina dos medidas:

* **TF (Term Frequency):** representa la frecuencia de un término dentro de un documento.
* **IDF (Inverse Document Frequency):** mide lo común o raro que es un término en el conjunto de documentos.

La frecuencia de término se calcula como:

$$
TF(t,d)=\frac{f_{t,d}}{\text{número total de palabras en }d}
$$

La frecuencia inversa del documento se calcula como:

$$
IDF(t,D)=\log\left(\frac{N}{df_t}\right)
$$

donde $N$ es el número total de documentos del corpus y $df_t$ es el número de documentos que contienen el término $t$.

El valor TF-IDF se obtiene multiplicando ambos componentes:

$$
TFIDF(t,d,D)=TF(t,d)\times IDF(t,D)
$$

El valor IDF es mayor para los términos que aparecen en pocos documentos y menor para los términos que aparecen en muchos documentos. De esta forma, TF-IDF permite distinguir entre términos comunes del corpus y términos más relevantes para determinados documentos.


In [8]:
N = matriz_bow.shape[0]

total_palabras_doc = matriz_bow.sum(dim=1, keepdim=True)
tf = matriz_bow / total_palabras_doc

df = (matriz_bow > 0).sum(dim=0)

idf = torch.log(torch.tensor(N, dtype=torch.float32) / df.float())

matriz_tfidf = tf * idf

print(f"Forma de la matriz TF-IDF: {matriz_tfidf.shape}")

Forma de la matriz TF-IDF: torch.Size([12, 411])


### Similitud del coseno con TF-IDF

Una vez obtenidos los vectores TF-IDF, se utiliza nuevamente `cosine_similarity` de PyTorch para calcular la similitud entre todos los pares de documentos.

In [9]:
similitud_tfidf = torch.zeros((N, N))

for i in range(N):
    for j in range(N):
        similitud_tfidf[i, j] = F.cosine_similarity(
            matriz_tfidf[i].unsqueeze(0),
            matriz_tfidf[j].unsqueeze(0)
        )

print("Matriz de similitud del coseno (TF-IDF):")

for i in range(N):
    fila = " ".join(f"{similitud_tfidf[i, j]:.2f}" for j in range(N))
    print(f"{nombres[i]}: {fila}")

Matriz de similitud del coseno (TF-IDF):
doc01.docx: 1.00 0.01 0.00 0.04 0.05 0.06 0.00 0.01 0.00 0.01 0.04 0.02
doc02.docx: 0.01 1.00 0.06 0.00 0.00 0.03 0.03 0.02 0.00 0.04 0.07 0.04
doc03.docx: 0.00 0.06 1.00 0.00 0.00 0.01 0.06 0.01 0.00 0.04 0.00 0.02
doc04.docx: 0.04 0.00 0.00 1.00 0.03 0.03 0.03 0.06 0.07 0.00 0.05 0.09
doc05.docx: 0.05 0.00 0.00 0.03 1.00 0.02 0.00 0.07 0.07 0.02 0.05 0.02
doc06.docx: 0.06 0.03 0.01 0.03 0.02 1.00 0.03 0.01 0.00 0.01 0.03 0.03
doc07.docx: 0.00 0.03 0.06 0.03 0.00 0.03 1.00 0.00 0.00 0.01 0.04 0.04
doc08.docx: 0.01 0.02 0.01 0.06 0.07 0.01 0.00 1.00 0.03 0.01 0.03 0.02
doc09.docx: 0.00 0.00 0.00 0.07 0.07 0.00 0.00 0.03 1.00 0.02 0.07 0.13
doc10.docx: 0.01 0.04 0.04 0.00 0.02 0.01 0.01 0.01 0.02 1.00 0.01 0.02
doc11.docx: 0.04 0.07 0.00 0.05 0.05 0.03 0.04 0.03 0.07 0.01 1.00 0.12
doc12.docx: 0.02 0.04 0.02 0.09 0.02 0.03 0.04 0.02 0.13 0.02 0.12 1.00


In [10]:
pares_tfidf = []

for i in range(N):
    for j in range(i + 1, N):
        pares_tfidf.append(
            (
                similitud_tfidf[i, j].item(),
                nombres[i],
                nombres[j]
            )
        )

pares_tfidf_ordenados = sorted(
    pares_tfidf,
    key=lambda x: x[0],
    reverse=True
)

print("Pares más similares (TF-IDF):")
for sim, a, b in pares_tfidf_ordenados[:5]:
    print(f"  {a} - {b}: {sim:.3f}")

print("\nPares menos similares (TF-IDF):")
for sim, a, b in pares_tfidf_ordenados[-5:]:
    print(f"  {a} - {b}: {sim:.3f}")

Pares más similares (TF-IDF):
  doc09.docx - doc12.docx: 0.132
  doc11.docx - doc12.docx: 0.117
  doc04.docx - doc12.docx: 0.091
  doc05.docx - doc08.docx: 0.072
  doc09.docx - doc11.docx: 0.071

Pares menos similares (TF-IDF):
  doc03.docx - doc09.docx: 0.000
  doc03.docx - doc11.docx: 0.000
  doc04.docx - doc10.docx: 0.000
  doc05.docx - doc07.docx: 0.000
  doc07.docx - doc09.docx: 0.000


In [11]:
print("Comparación de las 5 parejas más similares")
print("\nBoW:")
for sim, a, b in pares_ordenados[:5]:
    print(f"  {a} - {b}: {sim:.3f}")

print("\nTF-IDF:")
for sim, a, b in pares_tfidf_ordenados[:5]:
    print(f"  {a} - {b}: {sim:.3f}")

Comparación de las 5 parejas más similares

BoW:
  doc09.docx - doc12.docx: 0.323
  doc11.docx - doc12.docx: 0.302
  doc04.docx - doc12.docx: 0.249
  doc04.docx - doc09.docx: 0.217
  doc05.docx - doc09.docx: 0.207

TF-IDF:
  doc09.docx - doc12.docx: 0.132
  doc11.docx - doc12.docx: 0.117
  doc04.docx - doc12.docx: 0.091
  doc05.docx - doc08.docx: 0.072
  doc09.docx - doc11.docx: 0.071


## 8. Comparación entre BoW y TF-IDF

La representación Bag-of-Words utiliza la frecuencia de aparición de las palabras en cada documento. En cambio, TF-IDF tiene en cuenta dos componentes: la **frecuencia del término (TF)** dentro del documento y la **frecuencia inversa del documento (IDF)** dentro del corpus.

La TF se calcula como el número de veces que aparece un término en un documento dividido entre el número total de palabras de ese documento. La IDF se calcula como el logaritmo natural del número total de documentos dividido entre el número de documentos que contienen el término.
El valor TF-IDF se obtiene multiplicando ambos componentes.

De esta forma, los términos que aparecen en muchos documentos tienen un valor IDF menor, mientras que los términos que aparecen en pocos documentos tienen un valor IDF mayor. Por tanto, TF-IDF permite distinguir entre términos comunes del corpus y términos más relevantes para determinados documentos.

Al comparar los resultados obtenidos, la pareja con mayor similitud es la misma en ambos métodos: `doc09`–`doc12`. Con BoW obtiene una similitud de **0,323**, mientras que con TF-IDF obtiene **0,132**. La segunda pareja también coincide: `doc11`–`doc12`, con **0,302** mediante BoW y **0,117** mediante TF-IDF.

La tercera pareja es igualmente la misma en ambos métodos: `doc04`–`doc12`, con **0,249** en BoW y **0,091** en TF-IDF.

A partir de la cuarta posición aparecen diferencias. Con BoW, las siguientes parejas son `doc04`–`doc09` (0,217) y `doc05`–`doc09` (0,207). Con TF-IDF, en cambio, aparecen `doc05`–`doc08` (0,072) y `doc09`–`doc11` (0,071).

Por tanto, los dos métodos mantienen algunas de las mismas parejas entre los documentos con mayor similitud, especialmente `doc09`–`doc12`, `doc11`–`doc12` y `doc04`–`doc12`. Sin embargo, los valores y el orden de algunas parejas cambian al utilizar TF-IDF, debido a que esta representación tiene en cuenta tanto la frecuencia de los términos en cada documento como el número de documentos en los que aparecen.
